# Store-Item Demand Forecasting with LightGBM

This notebook demonstrates a complete workflow for store-item demand forecasting, including feature engineering, custom metrics, model training, and submission generation using LightGBM.

## 1. Import Libraries and Set Display Options
Import numpy, pandas, matplotlib, seaborn, and lightgbm. Set pandas display options for better readability.

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import lightgbm as lgb

pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

## 2. Load and Explore Data
Load train and test CSV files with date parsing. Concatenate datasets for unified processing. Display samples and info.

In [ ]:
train = pd.read_csv("datasets/demand_forecasting/train.csv", parse_dates=["date"])
test = pd.read_csv("datasets/demand_forecasting/test.csv", parse_dates=["date"])

df = pd.concat([train, test])

display(df.sample(5))
df.info()

## 3. Initial Data Analysis
Perform basic EDA: describe statistics, check date ranges, unique store/item counts, and groupby aggregations.

In [ ]:
display(df.describe().T)
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Unique stores:", df[["store"]].nunique().values[0])
print("Unique items:", df[["item"]].nunique().values[0])
display(df.groupby(["store"])["item"].nunique())
display(df.groupby(["store", "item"]).agg({"sales":[ "sum", "mean", "median", "std"]}))

## 4. Feature Engineering: Date Features
Create new date-based features (month, day_of_month, day_of_year, week_of_year, day_of_week, year, is_wknd, is_month_start, is_month_end) using a function.

In [ ]:
def create_date_features(dataframe):
    dataframe['month'] = dataframe.date.dt.month
    dataframe['day_of_month'] = dataframe.date.dt.day
    dataframe['day_of_year'] = dataframe.date.dt.dayofyear
    dataframe['week_of_year'] = dataframe.date.dt.weekofyear
    dataframe['day_of_week'] = dataframe.date.dt.dayofweek
    dataframe['year'] = dataframe.date.dt.year
    dataframe["is_wknd"] = dataframe.date.dt.weekday // 4
    dataframe['is_month_start'] = dataframe.date.dt.is_month_start.astype(int)
    dataframe['is_month_end'] = dataframe.date.dt.is_month_end.astype(int)
    return dataframe

df = create_date_features(df)
display(df.head())

## 5. Feature Engineering: Lag Features
Define and apply a function to create lag features for sales with added random noise for specified lag periods.

In [ ]:
def random_noise(dataframe):
    return np.random.normal(scale=1.6, size=(len(dataframe),))

df.sort_values(by=['store', 'item', 'date'], axis=0, inplace=True)

def lag_features(dataframe, lags):
    for lag in lags:
        dataframe['sales_lag_' + str(lag)] = dataframe.groupby(["store", "item"])["sales"].transform(
            lambda x: x.shift(lag)) + random_noise(dataframe)
    return dataframe

lags = [91, 98, 105, 112, 119, 126, 182, 364, 546, 728]
df = lag_features(df, lags)
display(df.head())

## 6. Feature Engineering: Rolling Mean Features
Define and apply a function to create rolling mean features for sales with triangular window and random noise.

In [ ]:
def roll_mean_features(dataframe, windows):
    for window in windows:
        dataframe['sales_roll_mean_' + str(window)] = dataframe.groupby(["store", "item"])["sales"]. \
            transform(lambda x: x.shift(1).rolling(window=window, min_periods=10, win_type="triang").mean()) + random_noise(dataframe)
    return dataframe

df = roll_mean_features(df, [365, 546])
display(df.head())

## 7. Feature Engineering: Exponential Weighted Mean Features
Define and apply a function to create EWM features for sales using multiple alphas and lag periods.

In [ ]:
alphas = [0.95, 0.9, 0.8, 0.7, 0.5]
lags = [91, 98, 105, 112, 180, 270, 365, 546, 728]

def ewm_features(dataframe, alphas, lags):
    for alpha in alphas:
        for lag in lags:
            dataframe['sales_ewm_alpha_' + str(alpha).replace(".", "") + "_lag_" + str(lag)] = \
                dataframe.groupby(["store", "item"])["sales"].transform(lambda x: x.shift(lag).ewm(alpha=alpha).mean())
    return dataframe

df = ewm_features(df, alphas, lags)
display(df.head())

## 8. One-Hot Encoding of Categorical Variables
Apply pd.get_dummies to store, item, day_of_week, and month columns.

In [ ]:
df = pd.get_dummies(df, columns=['store', 'item', 'day_of_week', 'month'])
display(df.head())
print(df.shape)

## 9. Log Transformation of Target Variable
Apply log1p transformation to the sales column to stabilize variance.

In [ ]:
df['sales'] = np.log1p(df["sales"].values)
display(df.head())

## 10. Custom SMAPE Metric Implementation
Define SMAPE and LightGBM-compatible SMAPE evaluation functions for model validation.

In [ ]:
def smape(preds, target):
    n = len(preds)
    masked_arr = ~((preds == 0) & (target == 0))
    preds, target = preds[masked_arr], target[masked_arr]
    num = np.abs(preds - target)
    denom = np.abs(preds) + np.abs(target)
    smape_val = (200 * np.sum(num / denom)) / n
    return smape_val

def lgbm_smape(preds, train_data):
    labels = train_data.get_label()
    smape_val = smape(np.expm1(preds), np.expm1(labels))
    return 'SMAPE', smape_val, False

## 11. Train-Validation Split
Split the data into training and validation sets based on date ranges. Prepare feature and target arrays.

In [ ]:
# Training: 2013-01-01 to 2016-12-31
# Validation: 2017-01-01 to 2017-03-31
train = df.loc[(df["date"] < "2017-01-01"), :]
val = df.loc[(df["date"] >= "2017-01-01") & (df["date"] < "2017-04-01"), :]

cols = [col for col in train.columns if col not in ['date', 'id', "sales", "year"]]

y_train = train['sales']
X_train = train[cols]

y_val = val['sales']
X_val = val[cols]

print(y_train.shape, X_train.shape, y_val.shape, X_val.shape)

## 12. Prepare LightGBM Datasets
Create LightGBM Dataset objects for training and validation using selected features.

In [ ]:
lgbtrain = lgb.Dataset(data=X_train, label=y_train, feature_name=cols)
lgbval = lgb.Dataset(data=X_val, label=y_val, reference=lgbtrain, feature_name=cols)

## 13. Train LightGBM Model with Early Stopping
Train a LightGBM model with specified parameters, early stopping, and custom SMAPE evaluation. Output training logs.

In [ ]:
lgb_params = {'num_leaves': 10,
              'learning_rate': 0.02,
              'feature_fraction': 0.8,
              'max_depth': 5,
              'verbose': 0,
              'num_boost_round': 10000,
              'early_stopping_rounds': 200,
              'nthread': -1}

model = lgb.train(lgb_params, lgbtrain,
                  valid_sets=[lgbtrain, lgbval],
                  num_boost_round=lgb_params['num_boost_round'],
                  early_stopping_rounds=lgb_params['early_stopping_rounds'],
                  feval=lgbm_smape,
                  verbose_eval=100)

## 14. Evaluate Validation Performance
Predict on validation set, compute SMAPE, and display the result.

In [ ]:
y_pred_val = model.predict(X_val, num_iteration=model.best_iteration)
val_smape = smape(np.expm1(y_pred_val), np.expm1(y_val))
print(f"Validation SMAPE: {val_smape:.4f}")

## 15. Feature Importance Visualization
Plot or print the top feature importances using model's gain metric.

In [ ]:
def plot_lgb_importances(model, plot=False, num=10):
    gain = model.feature_importance('gain')
    feat_imp = pd.DataFrame({'feature': model.feature_name(),
                             'split': model.feature_importance('split'),
                             'gain': 100 * gain / gain.sum()}).sort_values('gain', ascending=False)
    if plot:
        plt.figure(figsize=(10, 10))
        sns.set(font_scale=1)
        sns.barplot(x="gain", y="feature", data=feat_imp[0:25])
        plt.title('feature')
        plt.tight_layout()
        plt.show(block=True)
    else:
        display(feat_imp.head(num))
    return feat_imp

plot_lgb_importances(model, num=10, plot=True)

## 16. Retrain Model on Full Data
Retrain LightGBM model on all available training data using the best iteration from validation.

In [ ]:
train = df.loc[~df["sales"].isna()]
Y_train = train['sales']
X_train = train[cols]

test = df.loc[df.sales.isna()]
X_test = test[cols]

lgb_params = {'num_leaves': 10,
              'learning_rate': 0.02,
              'feature_fraction': 0.8,
              'max_depth': 5,
              'verbose': 0,
              'nthread': -1,
              "num_boost_round": model.best_iteration}

lgbtrain_all = lgb.Dataset(data=X_train, label=Y_train, feature_name=cols)
final_model = lgb.train(lgb_params, lgbtrain_all, num_boost_round=model.best_iteration)

## 17. Generate Test Predictions and Submission File
Predict on test set, inverse log transform predictions, prepare submission DataFrame, and save as CSV.

In [ ]:
test_preds = final_model.predict(X_test, num_iteration=model.best_iteration)

submission_df = test.loc[:, ["id", "sales"]]
submission_df['sales'] = np.expm1(test_preds)
submission_df['id'] = submission_df["id"].astype("int64")
submission_df.to_csv("submission_demand.csv", index=False)
submission_df.info()
display(submission_df.head())